# build-af3-from-scratch — Overview · 端到端 demo

本 notebook 把所有章节拼起来跑一遍完整的 AlphaFold 3 推理：加载自带的 7r6r 蛋白、加载 Protenix 官方权重、跑一次前向、输出预测结构 + pLDDT / pTM 分数。

End-to-end demo: load the bundled 7r6r protein, load the Protenix checkpoint, run inference, write the predicted structure, report pLDDT / pTM.

**前置**：先把 Protenix tiny 权重放到 `checkpoints/` 下：

```bash
mkdir -p checkpoints
curl -L -o checkpoints/protenix_tiny_default_v0.5.0.pt \
    https://protenix.tos-cn-beijing.volces.com/checkpoint/protenix_tiny_default_v0.5.0.pt
```

## 0. 基本环境 · Basic setup

Notebook 自动定位代码树根 (`solutions/` 或 `tutorials/`)。
学生在 `tutorials/` 中打开时，跑的就是 `tutorials/` 里自己填好的代码。

In [ ]:
import os, sys, time, json
from pathlib import Path

ROOTS = {'solutions', 'tutorials'}
if os.path.basename(os.getcwd()) not in ROOTS:
    while os.path.basename(os.getcwd()) not in ROOTS and os.getcwd() != '/':
        os.chdir('..')
    if os.path.basename(os.getcwd()) not in ROOTS:
        # Started at the repo root — prefer tutorials/ if present.
        if os.path.isdir('tutorials'):
            os.chdir('tutorials')
        elif os.path.isdir('solutions'):
            os.chdir('solutions')
assert os.path.basename(os.getcwd()) in ROOTS, (
    f'could not locate solutions/ or tutorials/ from {os.getcwd()}')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')

REPO_ROOT = Path(os.getcwd()).parent      # the parent of solutions/ or tutorials/
CKPT_DIR  = REPO_ROOT / 'checkpoints'
EXAMPLE   = Path(os.getcwd()) / 'examples' / 'example.json'
print('tree     =', os.getcwd())
print('ckpt dir =', CKPT_DIR, '(exists:', CKPT_DIR.is_dir(), ')')
print('example  =', EXAMPLE, '(exists:', EXAMPLE.is_file(), ')')

## 1. 构建模型 · Build the model

`Protenix` (`model/model.py`) 是顶层 `nn.Module`，把 InputFeatureEmbedder、Pairformer 主干、Diffusion、ConfidenceHead 拼起来 —— 论文 Algorithm 1。

In [ ]:
import torch
from copy import deepcopy
from configs.parser import parse_configs
from configs.configs_base import configs as base_cfg
from configs.configs_data import data_configs
from configs.configs_inference import inference_configs
from configs.configs_model_type import model_configs

MODEL_NAME = 'protenix_tiny_default_v0.5.0'

cfg = {**base_cfg, **{'data': data_configs}, **inference_configs}
cfg.update({
    'project': 'af3', 'run_name': 'overview', 'base_dir': '/tmp/af3',
    'eval_interval': 0, 'log_interval': 0,
    'input_json_path': str(EXAMPLE), 'model_name': MODEL_NAME,
    'triangle_attention': 'torch', 'triangle_multiplicative': 'torch',
    'enable_tf32': False, 'enable_efficient_fusion': False,
})
overrides = deepcopy(model_configs[MODEL_NAME])
def merge(d, s):
    for k, v in s.items():
        if isinstance(v, dict) and isinstance(d.get(k), dict): merge(d[k], v)
        else: d[k] = v
merge(cfg, overrides)
cfg = parse_configs(cfg, arg_str=None, fill_required_with_null=True)
cfg.model.N_cycle = 1            # quick demo: one trunk cycle
cfg.sample_diffusion.N_step   = 5  # 5 diffusion steps
cfg.sample_diffusion.N_sample = 1

from model.model import Protenix
try:
    model = Protenix(cfg).eval()
except (AttributeError, TypeError) as exc:
    # Almost always: some chapter's ``__init__`` is still ``pass`` so
    # nn.Module can't register the half-baked child. Give a hint.
    raise AssertionError(
        f'Protenix construction failed ({type(exc).__name__}: {exc}).\n'
        f'This usually means at least one chapter ``__init__`` is still '
        f'a ``pass`` stub. Finish the chapter labs first '
        f'(attention → pairformer → feature_embedding → diffusion → '
        f'confidence), then re-run overview.'
    ) from None
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Protenix built — {n_params:.2f} M parameters')

## 2. 加载权重 · Load the checkpoint

`load_state_dict(strict=False)` 会忽略 non-PLM Tiny 权重里残留的 `linear_esm.weight`。`missing=0 unexpected=1` 是预期的。

In [ ]:
# Newer torch refuses to unpickle argparse.Namespace under weights_only=True;
# 新版 torch 不允许未声明的 Namespace 反序列化，这里加白名单。
from argparse import Namespace
if hasattr(torch.serialization, 'add_safe_globals'):
    torch.serialization.add_safe_globals([Namespace])

ckpt_path = CKPT_DIR / f'{MODEL_NAME}.pt'
assert ckpt_path.is_file(), (
    f'missing checkpoint: {ckpt_path}\n'
    f'see README quickstart for the download command.')
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
state = ckpt['model'] if 'model' in ckpt else ckpt
state = {k.removeprefix('module.'): v for k, v in state.items()}
res = model.load_state_dict(state, strict=False)
print(f'missing={len(res.missing_keys)}  unexpected={len(res.unexpected_keys)}')
print('unexpected:', res.unexpected_keys)

## 3. 特征化 · Featurize the input

`get_inference_dataloader` 解析 JSON、跑 MSA / 模板 featurizer，输出可直接喂给模型的 `input_feature_dict`。

In [ ]:
from feature_extraction.inference.infer_dataloader import get_inference_dataloader
from runtime.torch_utils import to_device

loader = get_inference_dataloader(configs=cfg)
batch = next(iter(loader))
data, atom_array, err = batch[0]
assert not err, f'featurization failed: {err}'
print(f"sample: {data['sample_name']}")
print(f"  N_token = {int(data['N_token'])}")
print(f"  N_atom  = {int(data['N_atom'])}")
print(f"  N_msa   = {int(data['N_msa'])}")

## 4. 跑推理 · Run inference

一次前向 + 5 步 Euler diffusion 采样。在 M2 Max CPU 上约 10s。

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

model = model.to(device)
data  = to_device(data, device)

t0 = time.time()
with torch.no_grad():
    pred, _, _ = model(input_feature_dict=data['input_feature_dict'], mode='inference')
print(f'forward time: {time.time() - t0:.2f}s')

summary = pred['summary_confidence'][0]
print(f"  pLDDT         = {float(summary['plddt']):.2f}")
print(f"  pTM           = {float(summary['ptm']):.3f}")
print(f"  ranking score = {float(summary['ranking_score']):.3f}")
print(f"  has_clash     = {bool(summary['has_clash'])}")

## 5. 写预测结构 · Write the CIF

In [ ]:
from feature_extraction.utils import save_structure_cif

out_dir = REPO_ROOT / 'out_demo'
out_dir.mkdir(exist_ok=True)
cif_path = out_dir / '7r6r_pred.cif'

entity_poly_type = {
    k: v for k, v in data['entity_poly_type'].items() if v != 'non-polymer'
}
save_structure_cif(
    atom_array=atom_array,
    pred_coordinate=pred['coordinate'][0],
    output_fpath=str(cif_path),
    entity_poly_type=entity_poly_type,
    pdb_id='7r6r_pred',
)
print('wrote:', cif_path, f'({cif_path.stat().st_size // 1024} KB)')

## 6. 可视化 · Visualize

装了 `py3Dmol` 就能直接在 notebook 里渲染；否则用 ChimeraX / PyMOL 打开 `.cif`。

In [ ]:
try:
    import py3Dmol
    view = py3Dmol.view(width=600, height=400)
    with open(cif_path) as f:
        view.addModel(f.read(), 'mmcif')
    view.setStyle({'cartoon': {'color': 'spectrum'}})
    view.zoomTo()
    view.show()
except ImportError:
    print('py3Dmol not installed; skip in-notebook view.')
    print('Open', cif_path, 'in ChimeraX / PyMOL.')

## 接下来 · What next

- `solutions/<chapter>/<chapter>.ipynb` 把每章再走一遍，看每个空对应论文哪条算法。
- `python check_solutions.py` 一次跑完所有章节 (默认不含 overview)。
- `python generate_control_values.py --verify --src tutorials` 验证学生填空版。

If you opened this from `tutorials/`, the inference above just ran on
**your** implementation — congratulations on building AF3 from scratch.